# Laboratório Prático: Computação Paralela com OpenMP

**Disciplina:** Programação Concorrente
**Professor:** Gabriel P. Silva

**Duração:** ~2 horas

---

---
## Exercício 1: Visualização do Escalonamento e Distribuição de Laços
A diretiva `#pragma omp for` divide as iterações de um laço entre as threads disponíveis. O comportamento exato dessa divisão muda drasticamente conforme a cláusula `schedule` (escalonamento) escolhida.

Modifique o código abaixo testando os seguintes tipos de escalonamento na diretiva:
1. `schedule(static, 1)`
2. `schedule(static, 4)`
3. `schedule(dynamic)`
4. `schedule(dynamic, 3)`
5. `schedule(guided, 2)`

Observe e reporte como a ordem e o bloco de iterações executadas por cada *thread* muda em cada caso.

In [ ]:
%%writefile escalonamento.c
#include <stdio.h>
#include <omp.h>

int main() {
    // Defina a cláusula schedule no laço abaixo de acordo com o enunciado
    #pragma omp parallel for num_threads(4) schedule(static, 1)
    for (int i = 0; i < 16; i++) {
        printf("Iteração %2d executada pela thread %d\n", i, omp_get_thread_num());
    }
    return 0;
}

Overwriting escalonamento.c


In [ ]:
!gcc -O3 -fopenmp escalonamento.c -o escalonamento && ./escalonamento

---
## Exercício 2: Análise Teórico-Prática de Escopo (Shared vs Private)
Analise o código abaixo. Ele possui uma variável chamada `resultado` inicializada com o valor `100` fora da região paralela. Dentro da região paralela, cada thread incrementa essa variável com o seu próprio identificador (`tid`).

Sua tarefa é modificar a linha do `#pragma` para testar quatro cenários e responder: **Qual valor será impresso dentro e após a região paralela em cada caso?**

* **Cenário A:** Não declarar nenhuma cláusula (deixar o comportamento padrão).
* **Cenário B:** Declarar explicitamente como `shared(resultado)`.
* **Cenário C:** Declarar explicitamente como `private(resultado)`.
* **Cenário D:** Declarar explicitamente como `firstprivate(resultado)`.

In [ ]:
%%writefile teste_escopo.c
#include <stdio.h>
#include <omp.h>

int main() {
    int resultado = 100;

    // Altere esta linha para responder aos Cenários A, B, C e D
    #pragma omp parallel num_threads(4)
    {
        int tid = omp_get_thread_num();
        resultado = resultado + tid;
        printf("Valor local da variável 'resultado': %d\n", resultado);

    }

    printf("Valor final da variável 'resultado': %d\n", resultado);
    return 0;
}

Writing teste_escopo.c


In [ ]:
!gcc -o teste teste_escopo.c -fopenmp
!./teste

Valor local da variável 'resultado': 104
Valor local da variável 'resultado': 100
Valor local da variável 'resultado': 101
Valor local da variável 'resultado': 102
Valor final da variável 'resultado': 104


---
## Exercício 3: Correção de Escopo de Variáveis (Condição de Corrida)
O código abaixo tenta calcular o fatorial de pequenos valores armazenando os resultados parciais em uma variável auxiliar compartilhada por padrão. No entanto, há um erro grave de escopo que gera inconsistências (Condição de Corrida).

**Sua missão:**
1. Generalize o código para um valor N genérico.
2. Acrescente uma cláusula de paralelização condicional para que o laço seja paralelizado apenas se N >= 5.
3. Identifique qual(is) variável(is) está(ão) com o escopo incorreto e corrija a diretiva do OpenMP utilizando a cláusula correta (`private` ou `shared`) para que o programa funcione de forma consistente.
4. Qual a alternativa ao uso dessas diretivas para que o programa funcione corretamente?
5. Qual o valor final para as variáveis de `i` e `j` no final da região paralela?
6. Modifique o código para garantir que esses valores sejam iguais à execução sequencial do laço.

In [ ]:
%%writefile escopo_errado.c
#include <stdio.h>
#include <omp.h>

int main() {
    int i,j;
    long temp_fatorial;

    #pragma omp parallel for num_threads(4)
    for (i = 1; i <= 8; i++) {
        temp_fatorial = 1;
        for (j = 1; j <= i; j++) {
            temp_fatorial *= j;
        }
        printf("Fatorial de %d = %ld (pela thread %d)\n", i, temp_fatorial, omp_get_thread_num());
    }
    printf("\nValores finais fora do laco -> i: %d, j: %d\n", i, j);

    return 0;
}

In [ ]:
!gcc -fopenmp escopo_errado.cpp -o escopo_errado && ./escopo_errado

---
## Exercício 4: Exemplo Final Completo (Redução e Medição de Tempo)

Vamos agora analisar e paralelizar um problema real: aproximar o valor da constante matemática **$\pi$** através de integração numérica.

Abaixo está a implementação estritamente sequencial. Você deverá criar a versão paralela dela utilizando:
1. `#pragma omp parallel for` para paralelizar as iterações.
2. A cláusula `reduction` para acumular a soma com segurança eliminando condições de corrida.
3. A função `omp_get_wtime()` para medir o tempo de execução do trecho crítico e calcular o ganho de desempenho (Speedup) com 2, 4, 6 e 8 threads.

### Versão Sequencial (Apenas para Execução e Referência)

In [ ]:
%%writefile pi_sequencial.c
#include <stdio.h>
#include <time.h>

static long num_passos = 100000000;
double passo;

int main() {
    long i;
    double x, pi, soma = 0.0;
    struct timespec inicio, fim;

    passo = 1.0 / (double) num_passos;

    clock_gettime(CLOCK_MONOTONIC, &inicio);

    for (i = 0; i < num_passos; i++) {
        x = (i + 0.5) * passo;
        soma = soma + 4.0 / (1.0 + x*x);
    }
    pi = passo * soma;

    clock_gettime(CLOCK_MONOTONIC, &fim);
    double tempo = (fim.tv_sec - inicio.tv_sec) + (fim.tv_nsec - inicio.tv_nsec) / 1000000000.0;

    printf("Sqncl: O valor de PI é %f\n", pi);
    printf("Tempo gasto: %f segundos\n", tempo);
    return 0;
}

In [ ]:
!gcc -O3 pi_sequencial.cpp -o pi_sequencial && ./pi_sequencial

### Sua vez: Implemente a Versão Paralela
Complete o código com as funções `omp_get_wtime()`, inclua as diretivas adequadas e verifique se o valor calculado permanece idêntico enquanto que o tempo diminui.

In [ ]:
%%writefile pi_paralelo.c
#include <stdio.h>
#include <omp.h>

static long num_passos = 100000000;
double passo;

int main() {
    double x, pi, soma = 0.0;
    long i;

    passo = 1.0 / (double) num_passos;

    // 2. TODO: Aplicar '#pragma omp' e escopo correto das variáveis (só o necessário)
    for (i = 0; i < num_passos; i++) {
        x = (i + 0.5) * passo;
        soma = soma + 4.0 / (1.0 + x*x);
    }

    pi = passo * soma;

    printf("Paralelo: O valor de PI é %f\n", pi);
    printf("Tempo gasto: %f segundos\n", tempo_fim - tempo_inicio);
    return 0;
}

In [ ]:
!gcc -O3 -fopenmp pi_paralelo.cpp -o pi_paralelo && ./pi_paralelo